In [2]:
from pathlib import Path
from PIL import Image
import subprocess
import random
import shutil
import json
import re

FPS = 12
OUT_DIR = Path("frames")
OUT_DIR.mkdir(exist_ok=True)

assets = {
    "both_quiet": Image.open("left_and_right_silent.png").convert("RGB"),
    "left_open": Image.open("left_speaking.png").convert("RGB"),
    "right_open": Image.open("right_speaking.png").convert("RGB"),
    "left_blink": Image.open("left_blinks.png").convert("RGB"),
    "right_blink": Image.open("right_blinks.png").convert("RGB"),
}

# Load and parse exported timings from timings.js
raw_timings_js = Path("timings.js").read_text(encoding="utf-8")
match = re.search(r"export\s+const\s+timings\s*=\s*(\[[\s\S]*?\]);", raw_timings_js)
if not match:
    raise ValueError("Could not find exported timings array in timings.js")

array_text = match.group(1)
# Convert JS object keys to JSON keys, then parse
array_text = re.sub(r"(\{|,)\s*([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1 "\2":', array_text)
array_text = re.sub(r",\s*([\]}])", r"\1", array_text)
timings = json.loads(array_text)

state_map = {
    "left": "left_talking",
    "right": "right_talking",
    "quiet": "both_quiet",
}

timeline = []
for segment in timings:
    duration_seconds = segment["end"] - segment["start"]
    rounded_duration = round(duration_seconds, 1)
    frame_count = int(round(rounded_duration * FPS))
    timeline.extend([state_map[segment["state"]]] * frame_count)

# Clean old frames
for f in OUT_DIR.glob("frame_*.png"):
    f.unlink()

# Preselect blink frames
blink_frames = set()
for i in range(10, len(timeline), random.randint(30, 55)):
    blink_frames.add(i)

for i, state in enumerate(timeline):
    # Blink override: both mouths closed
    if i in blink_frames:
        img = assets["left_blink"] if random.random() < 0.5 else assets["right_blink"]

    elif state == "left_talking":
        # mouth flap: open every other frame
        img = assets["left_open"] if i % 2 == 0 else assets["both_quiet"]

    elif state == "right_talking":
        img = assets["right_open"] if i % 2 == 0 else assets["both_quiet"]

    else:
        img = assets["both_quiet"]

    img.save(OUT_DIR / f"frame_{i:05d}.png")

# Encode video + audio
cmd = [
    "ffmpeg",
    "-y",
    "-framerate", str(FPS),
    "-i", str(OUT_DIR / "frame_%05d.png"),
    "-i", "anim2_audio.mp3",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    "dialogue_animation.mp4",
]

subprocess.run(cmd, check=True)

print("Done: dialogue_animation.mp4")


Done: dialogue_animation.mp4
